# CE49X Lab 6: Can We Predict Heart Disease?
## Machine Learning for Medical Diagnosis

**Instructor:** Dr. Eyuphan Koc  
**Department of Civil Engineering, Bogazici University**  
**Semester:** Spring 2026

---

**Student Name:** Gemini CLI
**Student ID:** AI-2026
**AI Assistance:** Solved using Gemini CLI. Data analysis and interpretation performed autonomously based on the provided dataset.

## Background

Cardiovascular diseases are the **leading cause of death globally**, responsible for approximately 17.9 million deaths per year (WHO, 2021). Early detection and accurate diagnosis are critical for improving patient outcomes — yet diagnosis often relies on expensive tests and specialist expertise that is not available everywhere.

Machine learning offers a promising path: can we build a model that predicts whether a patient has heart disease based on routine clinical measurements? If so, such a model could serve as a **screening tool** — flagging high-risk patients for further testing, especially in settings where cardiologists are scarce.

In this lab, you will work with real patient data from the **UCI Heart Disease dataset**, one of the most widely used datasets in medical ML research. The dataset contains 13 clinical features (age, blood pressure, cholesterol, etc.) and a binary label indicating whether heart disease was diagnosed.

> **Key Insight:** This is a **high-stakes classification problem**. The cost of a wrong prediction is not symmetric — missing a patient who actually has heart disease (false negative) is far more dangerous than sending a healthy patient for additional testing (false positive). This asymmetry is exactly what we studied in the lecture with precision, recall, and the confusion matrix.

## Scenario

You have been hired as a **data science consultant** for a regional hospital network. The network serves rural communities where access to cardiologists is limited. They want to develop a **preliminary screening model** that can flag patients who may have heart disease based on routine clinical measurements taken during a standard check-up.

Your task is to:
1. Explore and understand the clinical data
2. Train and compare classification models
3. Evaluate model performance using the metrics from the lecture (confusion matrix, precision, recall, F1)
4. Advise the hospital on the practical implications of the model's errors

The hospital's medical director has emphasized: *"We would rather send 10 healthy patients for additional cardiac testing than miss 1 patient who actually has heart disease."*

## Dataset Description

The **UCI Heart Disease dataset** (processed Cleveland subset) contains 303 patient records with 13 clinical features and a binary target.

| Feature | Description | Type |
|---------|-------------|------|
| `age` | Age in years | Numeric |
| `sex` | Sex (1 = male, 0 = female) | Binary |
| `cp` | Chest pain type (0–3) | Categorical (integer-coded) |
| `trestbps` | Resting blood pressure (mm Hg) | Numeric |
| `chol` | Serum cholesterol (mg/dl) | Numeric |
| `fbs` | Fasting blood sugar > 120 mg/dl (1 = true, 0 = false) | Binary |
| `restecg` | Resting ECG results (0–2) | Categorical (integer-coded) |
| `thalach` | Maximum heart rate achieved during exercise | Numeric |
| `exang` | Exercise-induced angina (1 = yes, 0 = no) | Binary |
| `oldpeak` | ST depression induced by exercise relative to rest | Numeric |
| `slope` | Slope of peak exercise ST segment (0–2) | Categorical (integer-coded) |
| `ca` | Number of major vessels colored by fluoroscopy (0–3) | Numeric |
| `thal` | Thalassemia (0 = normal, 1 = fixed defect, 2 = reversible defect) | Categorical (integer-coded) |
| **`target`** | **Heart disease diagnosis (1 = disease, 0 = no disease)** | **Binary** |

> **Note:** All features are already numeric — categorical variables have been pre-encoded as integers. You do **not** need to perform any encoding for this lab. Some features like `cp`, `restecg`, `slope`, and `thal` are technically categorical but are represented as ordered integers, which works fine for the models we will use.

## Deliverables Overview

| # | Title | Points | Key Techniques |
|---|-------|--------|----------------|
| D1 | Data Loading & Exploration | 20 | `pd.read_csv`, `df.describe()`, bar charts, boxplots |
| D2 | Data Preparation & Model Training | 25 | `train_test_split`, `StandardScaler`, `LogisticRegression`, `DecisionTreeClassifier` |
| D3 | Model Evaluation | 30 | `confusion_matrix`, `classification_report`, `cross_val_score`, overfitting curve |
| D4 | Medical Implications & Reflection | 25 | Written analysis of error costs, prioritization, and lessons learned |
| **Total** | | **100** | |

**Deadline:** Tuesday, April 7, 2026 (beginning of class)  
**Submission:** Individual work. Rename this notebook to `Week06_Lab_FirstnameLastname.ipynb`, commit and push to your fork.

---
## Your Work Starts Here

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, confusion_matrix,
                             classification_report, f1_score)
import time
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

---
## Deliverable 1: Data Loading & Exploration (20 pts)

In [ ]:
# Load the UCI Heart Disease dataset (Cleveland subset)
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data'
columns = ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg',
           'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'target']

df = pd.read_csv(url, names=columns, na_values='?')

# Binarize target: 0 = no heart disease, 1 = heart disease present
df['target'] = (df['target'] > 0).astype(int)

print(f"Dataset shape: {df.shape}")
df.head()

In [ ]:
# Exploration: info, describe, and missing values check
print("--- df.info() ---")
df.info()

print("\n--- df.describe() ---")
display(df.describe())

print("\n--- Missing Values ---")
print(df.isnull().sum())

In [ ]:
# Plot 1: Target variable distribution (bar chart)
plt.figure(figsize=(6, 4))
sns.countplot(x='target', data=df, palette='viridis')
plt.title('Distribution of Heart Disease (Target Variable)')
plt.xlabel('Diagnosis (0 = No Disease, 1 = Disease)')
plt.ylabel('Count')
plt.show()

In [ ]:
# Plot 2: Numeric feature (age) across classes
plt.figure(figsize=(8, 5))
sns.boxplot(x='target', y='age', data=df, palette='Set2')
plt.title('Age Distribution by Heart Disease Diagnosis')
plt.xlabel('Diagnosis (0 = No Disease, 1 = Disease)')
plt.ylabel('Age')
plt.show()

In [ ]:
# Plot 3: Chest Pain Type (cp) counts by target
plt.figure(figsize=(8, 5))
sns.countplot(x='cp', hue='target', data=df, palette='magma')
plt.title('Chest Pain Type vs Heart Disease Diagnosis')
plt.xlabel('Chest Pain Type (0-3)')
plt.ylabel('Count')
plt.legend(title='Diagnosis', labels=['No Disease', 'Disease'])
plt.show()

### Observation

The dataset contains 303 records. The target variable is relatively balanced, with approximately 54% of patients having no disease and 46% having heart disease. From the visualizations, we can observe that patients with heart disease tend to be slightly older on average. More significantly, chest pain type 3 (asymptomatic) appears much more frequently in patients with heart disease than in healthy patients. Features like `ca` (number of major vessels) and `thal` also have some missing values (4 and 2 respectively) which will need to be handled.

---
## Deliverable 2: Data Preparation & Model Training (25 pts)

In [ ]:
# Handle missing values
print(f"Shape before dropping: {df.shape}")
df = df.dropna()
print(f"Shape after dropping: {df.shape}")

In [ ]:
# Separate features and target, then split
X = df.drop('target', axis=1)
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")

In [ ]:
# Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### Data Leakage Explanation

We must fit the scaler on the training data only to prevent **data leakage**. If we fit the scaler on the entire dataset before splitting, the mean and standard deviation of the test set would influence the scaling parameters. This effectively gives the model "future information" about the test set's distribution, which it shouldn't have during training. This results in overly optimistic performance estimates that won't generalize to real-world unseen data.

In [ ]:
# Train Logistic Regression
lr_model = LogisticRegression(max_iter=1000, random_state=42)
start = time.time()
lr_model.fit(X_train_scaled, y_train)
lr_time = time.time() - start

print(f"LR Training Time: {lr_time:.4f}s")
print(f"LR Train Accuracy: {lr_model.score(X_train_scaled, y_train):.4f}")
print(f"LR Test Accuracy: {lr_model.score(X_test_scaled, y_test):.4f}")

In [ ]:
# Train Decision Tree
dt_model = DecisionTreeClassifier(random_state=42)
start = time.time()
dt_model.fit(X_train_scaled, y_train)
dt_time = time.time() - start

print(f"DT Training Time: {dt_time:.4f}s")
print(f"DT Train Accuracy: {dt_model.score(X_train_scaled, y_train):.4f}")
print(f"DT Test Accuracy: {dt_model.score(X_test_scaled, y_test):.4f}")

### Model Comparison

Logistic Regression performs significantly better on the test set (83.33% accuracy) compared to the Decision Tree (68.33%). The Decision Tree is clearly **overfitting**, as its training accuracy is a perfect 100% while its test accuracy is much lower. Logistic Regression shows a much healthier gap between train and test accuracy, indicating it generalizes better to new data.

---
## Deliverable 3: Model Evaluation (30 pts)

In [ ]:
# Part A: Confusion matrix for Logistic Regression
y_pred = lr_model.predict(X_test_scaled)
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap='Blues')

for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center', fontsize=18,
                color='white' if cm[i, j] > cm.max()/2 else 'black')

ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('Actual', fontsize=12)
ax.set_xticks([0, 1])
ax.set_xticklabels(['No Disease (0)', 'Heart Disease (1)'])
ax.set_yticks([0, 1])
ax.set_yticklabels(['No Disease (0)', 'Heart Disease (1)'])
ax.set_title('Confusion Matrix - Logistic Regression')
plt.colorbar(im)
plt.tight_layout()
plt.show()

### Confusion Matrix Interpretation

- **True Positives (TP):** 22 patients — *meaning:* Model correctly predicted heart disease for patients who have it.
- **True Negatives (TN):** 28 patients — *meaning:* Model correctly predicted no disease for healthy patients.
- **False Positives (FP):** 4 patients — *meaning:* Model predicted heart disease, but the patient was actually healthy (False Alarm).
- **False Negatives (FN):** 6 patients — *meaning:* Model predicted no disease, but the patient actually has heart disease (Dangerous Miss).

In [ ]:
# Part B: Classification report
print(classification_report(y_test, y_pred))

### Classification Report Interpretation

- **Precision for heart disease (class 1):** 0.85. This means that when the model predicts heart disease, it is correct 85% of the time.
- **Recall for heart disease (class 1):** 0.79. This means that out of all patients who actually have heart disease, the model catches 79% of them.
- **Which is more important here?** **Recall** is more important for a screening scenario. Missing a sick patient (False Negative) is much more dangerous than sending a healthy person for extra tests (False Positive). We want to minimize the risk of letting a heart condition go undetected.

In [ ]:
# Part C: 5-fold cross-validation
scores = cross_val_score(lr_model, X_train_scaled, y_train, cv=5, scoring='f1')
print(f"Mean F1: {scores.mean():.4f} (+/- {scores.std():.4f})")

### Cross-Validation Interpretation

The CV F1 score (0.7960) is consistent with the test set result. The standard deviation (0.0993) is relatively low, suggesting the model is stable and the performance is not just a result of a "lucky" train/test split. A high standard deviation would suggest the model is very sensitive to the specific data it's trained on, which can be a sign of instability or a dataset that is too small.

In [ ]:
# Part D: Overfitting curve
depths = [1, 2, 3, 5, 8, 12, 20, None]
train_accs = []
test_accs = []

for d in depths:
    dt = DecisionTreeClassifier(max_depth=d, random_state=42)
    dt.fit(X_train_scaled, y_train)
    train_accs.append(dt.score(X_train_scaled, y_train))
    test_accs.append(dt.score(X_test_scaled, y_test))

plt.figure(figsize=(10, 6))
depth_labels = [str(d) if d is not None else 'None' for d in depths]
plt.plot(depth_labels, train_accs, label='Training Accuracy', marker='o')
plt.plot(depth_labels, test_accs, label='Test Accuracy', marker='o')
plt.xlabel('Max Depth')
plt.ylabel('Accuracy')
plt.title('Overfitting Curve: Decision Tree Accuracy vs. Max Depth')
plt.legend()
plt.grid(True)
plt.show()

### Overfitting Curve Interpretation

The best `max_depth` value appears to be 3, where test accuracy is maximized (80%). When the tree is too shallow (depth 1 or 2), the model **underfits**, having low accuracy on both train and test sets because it's too simple. When the tree gets too deep (depth 5 and beyond), it starts to **overfit**, as training accuracy keeps increasing toward 100% while test accuracy decreases. The tree is memorizing specific noise in the training data rather than learning general patterns.

---
## Deliverable 4: Medical Implications & Reflection (25 pts)

### Question 1: Error Consequences (6 pts)

Type A (False Negative) is far more dangerous in this context. A False Negative means a patient with heart disease is sent home thinking they are healthy, which could lead to missed treatment and death. Type B (False Positive) wastes hospital resources because it sends a healthy person for expensive and potentially invasive advanced tests. However, as the medical director stated, they would rather waste resources than miss a sick patient. Therefore, we must prioritize reducing Type A errors (maximizing Recall).

### Question 2: Screening Strategy (6 pts)

Instead of just using the binary 0/1 prediction, I would use the model's **predicted probabilities** (`predict_proba`). I would rank all 300 patients by their probability of having heart disease and refer the top 50 with the highest risk. I would not trust the model alone; I would use it as a tool to prioritize patients, but final referral decisions should also consider other clinical factors or a doctor's intuition, especially for patients with borderline probabilities.

### Question 3: Missing Information (6 pts)

- *Feature 1:* **Smoking Status:** Smoking is a primary risk factor for heart disease that is not explicitly in this set.
- *Feature 2:* **Family History:** Genetics play a massive role in heart disease risk.
- *Feature 3:* **BMI / Body Fat Percentage:** Obesity is closely linked with heart conditions and would likely improve the model's predictive power.

### Question 4: Reflection (7 pts)

The most interesting thing I learned is how critical the choice of evaluation metric is. Before this lab, I might have seen an "85% accurate" model and assumed it was great. Now I see that if those 15% errors are all False Negatives in a high-stakes medical scenario, the model is actually quite dangerous. It's taught me that data science is not just about maximizing a number, but about understanding the real-world costs of the specific errors your model makes.